In [0]:
# Define paths
silver_path = "/Volumes/workspace/default/olist_raw/silver/"
gold_path = "/Volumes/workspace/default/olist_raw/gold/"

# Load silver master table
silver = spark.read.format("delta").load(silver_path + "master")

print(f"Silver master loaded: {silver.count()} rows, {len(silver.columns)} columns")

Silver master loaded: 115361 rows, 38 columns


In [0]:
from pyspark.sql.functions import avg, count, sum, round, desc

# Gold table 1 — Sales by product category
sales_by_category = silver \
    .groupBy("product_category") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_revenue"),
        round(avg("price"), 2).alias("avg_order_value"),
        round(avg("review_score"), 2).alias("avg_review_score")
    ) \
    .orderBy(desc("total_revenue"))

sales_by_category.write.format("delta").mode("overwrite").save(gold_path + "sales_by_category")

print("Gold table 1 done — Sales by category")
sales_by_category.show(10)

Gold table 1 done — Sales by category
+--------------------+------------+-------------+---------------+----------------+
|    product_category|total_orders|total_revenue|avg_order_value|avg_review_score|
+--------------------+------------+-------------+---------------+----------------+
|       health_beauty|        9778|   1272531.66|         130.14|            4.18|
|       watches_gifts|        6073|   1213801.56|         199.87|            4.07|
|      bed_bath_table|       11745|   1087080.31|          92.56|            3.92|
|      sports_leisure|        8758|    993054.02|         113.39|            4.17|
|computers_accesso...|        7925|    921996.81|         116.34|             4.0|
|     furniture_decor|        8587|    748050.13|          87.11|            3.97|
|          housewares|        7185|    649935.54|          90.46|            4.12|
|          cool_stuff|        3911|    635063.53|         162.38|            4.19|
|                auto|        4289|     603374.1|

In [0]:
# Gold table 2 — Delivery performance by seller state
delivery_performance = silver \
    .groupBy("seller_state") \
    .agg(
        count("order_id").alias("total_orders"),
        round(avg("delivery_days"), 1).alias("avg_delivery_days"),
        round(avg("delivery_on_time") * 100, 1).alias("on_time_delivery_pct"),
        round(avg("review_score"), 2).alias("avg_review_score")
    ) \
    .orderBy(desc("on_time_delivery_pct"))

delivery_performance.write.format("delta").mode("overwrite").save(gold_path + "delivery_performance")

print("Gold table 2 done — Delivery performance by seller state")
delivery_performance.show(10)

Gold table 2 done — Delivery performance by seller state
+------------+------------+-----------------+--------------------+----------------+
|seller_state|total_orders|avg_delivery_days|on_time_delivery_pct|avg_review_score|
+------------+------------+-----------------+--------------------+----------------+
|          PI|          11|             13.7|               100.0|            4.36|
|          SE|          10|             12.6|               100.0|             3.9|
|          RO|          14|             17.4|               100.0|            4.08|
|          GO|         537|             12.7|                96.3|            4.32|
|          PE|         462|             12.8|                96.1|            4.14|
|          RS|        2255|             11.5|                95.6|            4.25|
|          PB|          43|             11.6|                95.3|            3.95|
|          MT|         146|             14.7|                95.2|             4.2|
|          MG|     

In [0]:
# Gold table 3 — Revenue and payment analysis by payment type
payment_analysis = silver \
    .groupBy("payment_type") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("payment_value"), 2).alias("total_revenue"),
        round(avg("payment_value"), 2).alias("avg_payment_value"),
        round(avg("payment_installments"), 1).alias("avg_installments")
    ) \
    .orderBy(desc("total_revenue"))

payment_analysis.write.format("delta").mode("overwrite").save(gold_path + "payment_analysis")

print("Gold table 3 done — Payment analysis")
payment_analysis.show()

Gold table 3 done — Payment analysis
+------------+------------+-------------+-----------------+----------------+
|payment_type|total_orders|total_revenue|avg_payment_value|avg_installments|
+------------+------------+-------------+-----------------+----------------+
| credit_card|       85124|1.523388246E7|           178.96|             3.6|
|      boleto|       22420|    3952673.2|            176.3|             1.0|
|     voucher|        6156|    396873.07|            64.47|             1.0|
|  debit_card|        1658|    247102.73|           149.04|             1.0|
|        NULL|           3|         NULL|             NULL|            NULL|
+------------+------------+-------------+-----------------+----------------+



In [0]:
# Gold table 4 — Customer satisfaction by product category
customer_satisfaction = silver \
    .groupBy("product_category") \
    .agg(
        count("order_id").alias("total_orders"),
        round(avg("review_score"), 2).alias("avg_review_score"),
        round(avg("delivery_days"), 1).alias("avg_delivery_days"),
        round(avg("freight_value"), 2).alias("avg_freight_value"),
        round(avg("price"), 2).alias("avg_price")
    ) \
    .orderBy("avg_review_score")

customer_satisfaction.write.format("delta").mode("overwrite").save(gold_path + "customer_satisfaction")

print("Gold table 4 done — Customer satisfaction by category")
customer_satisfaction.show(10)

Gold table 4 done — Customer satisfaction by category
+--------------------+------------+----------------+-----------------+-----------------+---------+
|    product_category|total_orders|avg_review_score|avg_delivery_days|avg_freight_value|avg_price|
+--------------------+------------+----------------+-----------------+-----------------+---------+
|security_and_serv...|           2|             2.5|             15.0|            20.61|   141.64|
| diapers_and_hygiene|          37|            3.38|             10.6|            14.74|    40.56|
|    office_furniture|        1760|            3.55|             20.9|            39.77|   159.13|
|      home_comfort_2|          31|            3.64|             14.3|            13.59|    24.94|
|fashion_male_clot...|         138|            3.65|             13.1|            16.32|    81.78|
|     fixed_telephony|         262|            3.76|             12.8|            17.65|   214.02|
|               audio|         378|            3.85|   